In [1]:
"""
Recruitment funnel generator - v2
Changes vs v1 (all marked with  # V2  comments):
  1. Source now affects conversion (SOURCE_MULT), normalised so the overall funnel stays ~1000>600>300>100>80
  2. Recruiters differ in speed (RECRUITER_GAP_MULT)
  3. final_status + rejected_at_stage, plus a fixed extract date so no stage date is in the future
  4. experience_years and gender added (gender has a DELIBERATE, adjustable disparity)
  5. Duplicate pairs: some deliberately carry a different TRUE source
Messiness kept: source casing/spacing, department whitespace, corrupted ages, duplicate ids.
"""
import pandas as pd, numpy as np, random
from faker import Faker

SEED = 42
random.seed(SEED); np.random.seed(SEED)
fake = Faker(); Faker.seed(SEED)

N_APPLICANTS = 1500                                # V2: bigger sample so source effects are detectable
EXTRACT_DATE = pd.Timestamp("2026-09-16")          # V2: fixed "data pulled on" date (reproducible)
START_DATE = EXTRACT_DATE - pd.Timedelta(days=365)
IN_PROCESS_WINDOW_DAYS = 30                        # V2: no activity for 30+ days => treated as Rejected

# ---------------- V2: sources -> canonical + messy variants + share + conversion multiplier
SOURCES = {
    #  canonical          share  mult   messy variants
    "LinkedIn":        (0.24, 1.20, ["LinkedIn", "linkedin", "LINKEDIN", "Linked in"]),
    "Referral":        (0.14, 2.20, ["Referral", "referral", "REFERRAL"]),
    "Indeed":          (0.17, 1.00, ["Indeed", "indeed", "Indeed "]),
    "Company Website": (0.15, 0.70, ["Company Website", "company website", "Company website"]),
    "Job Fair":        (0.15, 0.40, ["Job Fair", "job fair", "JobFair"]),
    "Naukri":          (0.15, 0.90, ["Naukri", "naukri", "NAUKRI"]),
}
src_names = list(SOURCES)
src_shares = np.array([SOURCES[s][0] for s in src_names]); src_shares /= src_shares.sum()
# normalise multipliers so the share-weighted average is exactly 1 (keeps overall funnel on target)
norm = sum(share * SOURCES[s][1] for s, share in zip(src_names, src_shares))
SOURCE_MULT = {s: SOURCES[s][1] / norm for s in src_names}

job_roles = ["Data Analyst", "Software Engineer", "Product Manager", "HR Executive",
             "Sales Associate", "Marketing Specialist", "Business Analyst",
             "QA Engineer", "DevOps Engineer", "UX Designer"]
departments = ["Engineering", "Engineering ", "Sales", "Marketing", "Marketing ",
               "Human Resources", "Finance", "Finance ", "Product", "Operations"]
recruiters = ["Aisha Khan", "Rohan Mehta", "Priya Sharma", "David Lee",
              "Sarah Johnson", "Vikram Singh", "Emily Chen", "Arjun Patel"]

# V2: recruiter speed (multiplies the days between stages)
RECRUITER_GAP_MULT = {"Rohan Mehta": 1.6, "Arjun Patel": 1.35, "Priya Sharma": 0.8}  # others = 1.0

# V2: DELIBERATE planted disparity for the fairness check (AI #5). Set to 1.0 to switch off.
# Document this in the README as a design choice.
FEMALE_CONVERSION_FACTOR = 0.75
GENDERS, GENDER_P = ["Male", "Female", "Non-binary"], [0.52, 0.45, 0.03]

FUNNEL_STAGES = ["screened_date", "interviewed_date", "offered_date", "hired_date"]
STAGE_LABEL = {"applied_date": "Applied", "screened_date": "Screened",
               "interviewed_date": "Interviewed", "offered_date": "Offered"}
STAGE_DAY_GAPS = {"screened_date": (2, 7), "interviewed_date": (3, 14),
                  "offered_date": (2, 10), "hired_date": (5, 21)}
BASE_THRESHOLDS = {"screened_date": 600, "interviewed_date": 300,
                   "offered_date": 100, "hired_date": 80}

# ---------------- Step 1: base rows
rows = []
for i in range(1, N_APPLICANTS + 1):
    canon = np.random.choice(src_names, p=src_shares)                    # V2: pick TRUE source first
    age = max(18, min(int(np.random.normal(29, 6)), 60))
    exp = round(max(0.0, (age - 22) - random.uniform(0, 4)), 1)          # V2
    rows.append({
        "applicant_id": i, "name": fake.name(), "age": age,
        "source": random.choice(SOURCES[canon][2]),                      # messy label of the true source
        "job_role": random.choice(job_roles), "department": random.choice(departments),
        "recruiter": random.choice(recruiters),
        "applied_date": START_DATE + pd.Timedelta(days=random.randint(0, 365)),
        "experience_years": exp,
        "gender": np.random.choice(GENDERS, p=GENDER_P),
        "_true_source": canon,
    })
df = pd.DataFrame(rows)

# V2: normalise the gender and experience factors so they shift WHO converts without shrinking the whole funnel
def _exp_factor(e): return 1 + 0.03 * min(max(e - 5, -4), 6)
EXP_NORM = df["experience_years"].map(_exp_factor).mean()
GENDER_NORM = sum(p * (FEMALE_CONVERSION_FACTOR if g == "Female" else 1.0) for g, p in zip(GENDERS, GENDER_P))

# ---------------- Step 2: funnel progression
for s in FUNNEL_STAGES: df[s] = pd.NaT
df["final_status"] = ""; df["rejected_at_stage"] = ""
draws = np.random.randint(1, 1001, size=len(df))

for idx, draw in enumerate(draws):
    r = df.loc[idx]
    mult = SOURCE_MULT[r["_true_source"]]
    mult *= _exp_factor(r["experience_years"]) / EXP_NORM                # V2: experience helps a little
    mult *= (FEMALE_CONVERSION_FACTOR if r["gender"] == "Female" else 1.0) / GENDER_NORM
    gap_mult = RECRUITER_GAP_MULT.get(r["recruiter"], 1.0)

    current, last_label, truncated = r["applied_date"], "Applied", False
    for stage in FUNNEL_STAGES:
        if draw <= min(1000, BASE_THRESHOLDS[stage] * mult):
            lo, hi = STAGE_DAY_GAPS[stage]
            new_date = current + pd.Timedelta(days=max(1, round(random.randint(lo, hi) * gap_mult)))
            if new_date > EXTRACT_DATE:          # V2: event would be in the future -> still in process
                truncated = True; break
            df.at[idx, stage] = new_date
            current = new_date
            last_label = STAGE_LABEL.get(stage, "Hired")
        else:
            break

    # V2: final status
    if last_label == "Hired":
        df.at[idx, "final_status"] = "Hired"
    elif truncated or (EXTRACT_DATE - current).days < IN_PROCESS_WINDOW_DAYS:
        df.at[idx, "final_status"] = "In Process"
    else:
        df.at[idx, "final_status"] = "Rejected"
        df.at[idx, "rejected_at_stage"] = last_label

# ---------------- Step 3: messiness
# 3a. duplicates - V2: 8 of 15 carry a different TRUE source, the rest get mild noise
n_dup = 15
dup_idx = np.random.choice(df.index, size=n_dup, replace=False)
dup = df.loc[dup_idx].copy()
for k, i in enumerate(dup.index):
    if k < 8:
        other = random.choice([s for s in src_names if s != dup.at[i, "_true_source"]])
        dup.at[i, "source"] = random.choice(SOURCES[other][2])
    elif random.random() < 0.5:
        dup.at[i, "source"] = random.choice(SOURCES[dup.at[i, "_true_source"]][2])
    if random.random() < 0.3:
        dup.at[i, "name"] = dup.at[i, "name"] + " "
df = pd.concat([df, dup], ignore_index=True)

# 3b. corrupted ages
for i in np.random.choice(df.index, size=12, replace=False):
    df.at[i, "age"] = random.choice([-5, -1, 0, 150, 200, 999])

# 3c. department trailing whitespace
for i in np.random.choice(df.index, size=40, replace=False):
    v = df.at[i, "department"]
    if isinstance(v, str) and not v.endswith(" "): df.at[i, "department"] = v + " "

# ---------------- Step 4: export
df = df.drop(columns=["_true_source"]).sample(frac=1, random_state=SEED).reset_index(drop=True)
for c in ["applied_date"] + FUNNEL_STAGES:
    df[c] = pd.to_datetime(df[c]).dt.strftime("%Y-%m-%d")
cols = ["applicant_id", "name", "age", "gender", "experience_years", "source", "job_role",
        "department", "recruiter", "applied_date"] + FUNNEL_STAGES + ["final_status", "rejected_at_stage"]
df[cols].to_csv("recruitment_funnel_v2.csv", index=False)
print(len(df), "rows saved")

1515 rows saved
